In [ ]:
# ===============================
# MOUNT DRIVE
# ===============================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ===============================
# IMPORTS
# ===============================
import cv2
import numpy as np
import os
import pandas as pd
from scipy.special import gamma

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from skimage.measure import shannon_entropy

np.random.seed(42)

# ===============================
# PATHS
# ===============================
image_folder = "/content/drive/MyDrive/Colab Notebooks/data/low_quality_images"
output_folder = "/content/drive/MyDrive/Colab Notebooks/results/output_ICCBF"

os.makedirs(output_folder, exist_ok=True)

# ===============================
# METRICS (UPDATED ONLY)
# ===============================
def compute_metrics(original, processed):

    original = original.astype(np.float32)
    processed = processed.astype(np.float32)

    ssim_val = ssim(original, processed, data_range=255)
    psnr_val = psnr(original, processed, data_range=255)
    mse_val = mse(original, processed)

    # NEW
    mae_val = np.mean(np.abs(original - processed))
    entropy_val = shannon_entropy(processed)

    edges_orig = cv2.Canny(original.astype(np.uint8), 100, 200)
    edges_proc = cv2.Canny(processed.astype(np.uint8), 100, 200)

    if np.sum(edges_orig) == 0:
        epi_val = 0
    else:
        epi_val = np.sum(edges_orig & edges_proc) / np.sum(edges_orig)

    return ssim_val, psnr_val, mse_val, mae_val, entropy_val, epi_val

# ===============================
# ENHANCEMENT (CLAHE + BILATERAL)
# ===============================
def enhance_image(img, clip, tile, d, sigmaColor, sigmaSpace):

    clahe = cv2.createCLAHE(
        clipLimit=float(clip),
        tileGridSize=(int(tile), int(tile))
    )
    cl = clahe.apply(img)

    bf = cv2.bilateralFilter(
        cl,
        int(d),
        sigmaColor,
        sigmaSpace
    )

    return bf

# ===============================
# FITNESS FUNCTION
# ===============================
def get_fitness(img):

    def f(p):
        clip, tile, d, sigmaColor, sigmaSpace = p

        enhanced = enhance_image(img, clip, tile, d, sigmaColor, sigmaSpace)

        ssim_val = ssim(img, enhanced, data_range=255)
        psnr_val = psnr(img, enhanced, data_range=255)
        mse_val = mse(img, enhanced)

        return (0.4 * ssim_val) + (0.4 * psnr_val) - (0.2 * mse_val)

    return f

# ===============================
# ICS CLASS
# ===============================
class ICS:
    def __init__(self, fitness, bounds, n=10, pa=0.25, beta=1.5, iters=8):
        self.fit = fitness
        self.bounds = np.array(bounds)
        self.n = n
        self.pa = pa
        self.beta = beta
        self.iters = iters
        self.dim = len(bounds)

        self.pop = np.random.uniform(self.bounds[:,0], self.bounds[:,1], (n,self.dim))
        self.fvals = np.array([self.fit(x) for x in self.pop])

    def levy(self):
        b = self.beta
        sigma = (gamma(1+b)*np.sin(np.pi*b/2)/(gamma((1+b)/2)*b*2**((b-1)/2)))**(1/b)
        u = np.random.normal(0, sigma, self.dim)
        v = np.random.normal(0, 1, self.dim)
        return u/(np.abs(v)**(1/b))

    def clip(self, x):
        return np.clip(x, self.bounds[:,0], self.bounds[:,1])

    def run(self):
        for _ in range(self.iters):

            for i in range(self.n):
                new = self.clip(self.pop[i] + 0.03 * self.levy())
                fnew = self.fit(new)

                j = np.random.randint(self.n)
                if fnew > self.fvals[j]:
                    self.pop[j] = new
                    self.fvals[j] = fnew

            for i in range(self.n):
                if np.random.rand() < self.pa:
                    self.pop[i] = np.random.uniform(
                        self.bounds[:,0],
                        self.bounds[:,1],
                        self.dim
                    )
                    self.fvals[i] = self.fit(self.pop[i])

        return self.pop[np.argmax(self.fvals)]

# ===============================
# MAIN LOOP
# ===============================
files = [f for f in os.listdir(image_folder)
         if f.lower().endswith((".jpg",".png",".jpeg"))][:25]

records = []

bounds = [
    (1.0, 3.0),
    (6, 10),
    (5, 9),
    (30, 100),
    (30, 100)
]

for file in files:

    img = cv2.imread(os.path.join(image_folder, file))
    gray = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (256,256))

    ics = ICS(get_fitness(gray), bounds)
    best = ics.run()

    enhanced = enhance_image(gray, *best)

    cv2.imwrite(os.path.join(output_folder, file), enhanced)

    s, p, m, mae, ent, epi = compute_metrics(gray, enhanced)

    records.append([file, s, p, m, mae, ent, epi])

    print("Processed:", file, "Best:", best)

# ===============================
# SAVE RESULTS
# ===============================
df = pd.DataFrame(records, columns=[
    "Image", "SSIM", "PSNR", "MSE", "MAE", "Entropy", "EPI"
])

df.to_csv("/content/drive/MyDrive/Colab Notebooks/results/output_ICCBF/results.csv", index=False)

print("\nAverage Metrics:\n", df.mean(numeric_only=True))
print("FINAL BEST METHOD DONE 🚀")

Mounted at /content/drive
Processed: 00 (74).jpg Best: [ 1.00329955 10.          8.33722431 45.50730426 42.98632471]
Processed: 00 (139).jpg Best: [ 1.00984037  9.85794436  6.31814807 41.75858332 85.90253878]
Processed: 00 (84).jpg Best: [ 1.19613472  8.54613627  8.2267546  30.0307957  90.23008824]
Processed: 00 (137).jpg Best: [ 1.06238608  6.61609259  5.71878559 31.18428946 62.5215307 ]
Processed: 00 (59).jpg Best: [ 1.08261414  6.81817476  5.12824618 82.12891286 80.62971429]
Processed: 00 (112).jpg Best: [ 1.04095501  6.46082115  7.36835491 60.1885902  83.58160728]
Processed: 00 (79).jpg Best: [ 1.          6.34409803  7.25596056 45.88511218 66.13570021]
Processed: 00 (57).jpg Best: [ 1.02497068  7.46431946  6.80272867 32.07750531 84.04916976]
Processed: 00 (132).jpg Best: [ 1.10804374  7.65811924  7.27527822 43.07440942 61.22192725]
Processed: 00 (81).jpg Best: [ 1.02563804  6.11373319  8.91460327 43.22041733 87.40684556]
Processed: 00 (111).jpg Best: [ 1.11970745  9.52586495  5.34